In [26]:
import requests
import pandas as pd

In [4]:
all_trxns = pd.read_csv("all_trxns.csv", dtype={"counterparty": str})

In [18]:
data_trn = all_trxns
data_trn["timestamp"] = pd.to_datetime(
    data_trn["timestamp"], format="%Y-%m-%d %H:%M:%S"
)
data_trn["date"] = data_trn["timestamp"].dt.date

In [20]:
# Set up the base URL and parameters that don't change
base_url = "https://api.apilayer.com/exchangerates_data/"

In [ ]:
import os
# Create an empty list to store the results
results = []

# Set the API key in the headers
# FOC-181 resolved 2026-09-02: the key string in git history is a dummy, not a real
# credential (Mateusz’s confirmation) — no rotation/purge needed. Key is loaded from env only.
headers = {"apikey": os.environ.get("APILAYER_KEY", "")}

# Iterate over the unique dates and currencies
for date in data_trn["date"].unique():
    date_str = str(date)  # Convert the date to a string
    ccy_list = list(data_trn.loc[data_trn["date"] == date, "ccy"].unique())
    symbols = ",".join(ccy_list)
    base = "EUR"
    url = f"https://api.apilayer.com/exchangerates_data/{date_str}?symbols={symbols}&base={base}"
    response = requests.get(url, headers=headers)
    data = response.json()  # Parse the response into a dictionary

    # Process the response data and add it to the results list
    rates = data["rates"]
    for currency, rate in rates.items():
        row = {"date": date, "ccy": currency, "rate": rate}
        results.append(row)

# Convert the results list to a DataFrame
df_results = pd.DataFrame(results)

In [24]:
# Convert the results list to a DataFrame
df_results = pd.DataFrame(results)

In [ ]:
df_results